In [1]:
!pip uninstall qdrant-client -y
!pip install qdrant-client==1.9.1

Found existing installation: qdrant-client 1.18.0
Uninstalling qdrant-client-1.18.0:
  Successfully uninstalled qdrant-client-1.18.0
  Using cached qdrant_client-1.9.1-py3-none-any.whl.metadata (9.5 kB)
Using cached qdrant_client-1.9.1-py3-none-any.whl (229 kB)


In [2]:
pip install -U qdrant-client sentence-transformers pymupdf

  Using cached qdrant_client-1.18.0-py3-none-any.whl.metadata (11 kB)
Using cached qdrant_client-1.18.0-py3-none-any.whl (398 kB)
  Attempting uninstall: qdrant-client
    Found existing installation: qdrant-client 1.9.1
    Uninstalling qdrant-client-1.9.1:
      Successfully uninstalled qdrant-client-1.9.1
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import fitz  # PyMuPDF
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.http.models import VectorParams, Distance, PointStruct
from dotenv import load_dotenv

# OPEN PDF
load_dotenv()

# CONNECT QDRANT
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
QDRANT_URL = os.getenv("QDRANT_URL")

client = QdrantClient(
    url = QDRANT_URL,
    api_key = QDRANT_API_KEY,
    timeout=60,
    check_compatibility=False
)

# COLLECTION NAME
pdf = fitz.open("50_page_product_catalog.pdf")

# LOAD EMBEDDING MODEL
model = SentenceTransformer("all-MiniLM-L6-v2")

collection_name = "clothing_collections"

# CREATE COLLECTION
if not client.collection_exists(collection_name):
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(
            size=384,
            distance=Distance.COSINE 
        )
    )

# STORE ALL POINTS
points = []

# LOOP THROUGH PDF PAGES
for page_num in range(len(pdf)):
    page = pdf[page_num]
    text = page.get_text().strip()

    # CREATE EMBEDDING
    embedding = model.encode(text)

    # CREATE POINT
    point = PointStruct(
        id=page_num,
        vector=embedding.tolist(),
        payload={
            "page": page_num + 1,
            "text": text
        }
    )

    points.append(point)

# UPSERT INTO QDRANT
client.upsert(
    collection_name=collection_name,
    points=points
)

print("\n✅ ALL CLOTHING DATA STORED IN QDRANT")

# SEARCH QUERY
query = "black shirt"
query_embedding = model.encode(query)

# SEARCH
result = client.query_points(
    collection_name=collection_name,
    query=query_embedding.tolist(),
    limit=3
).points

print("\n🔍 SEARCH RESULTS:\n")

# SHOW RESULTS
fields = [
    "Name",
    "Brand",
    "Price",
    "Color",
    "Sizes",
    "Season",
    "Stock",
    "Minimum Age",
    "Description"
]

for res in result:

    print("Score :", round(res.score, 2))
    print("Page  :", res.payload["page"])
    print()

    text = res.payload["text"]

    lines = [line.strip() for line in text.split("\n") if line.strip()]

    # REMOVE UNWANTED LINES
    cleaned_lines = [
        line for line in lines
        if "Product Catalog Page" not in line
        and line not in ["Field", "Value"]
    ]

    # FORMAT OUTPUT
    for i in range(len(cleaned_lines)-1):

        if cleaned_lines[i] in fields:

            key = cleaned_lines[i]
            value = cleaned_lines[i+1]

            print(f"{key} : {value}")

    print("\n----------------------")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


✅ ALL CLOTHING DATA STORED IN QDRANT

🔍 SEARCH RESULTS:

Score : 0.57
Page  : 1

Name : Black Shirt
Brand : Puma
Price : I1968
Color : Black
Sizes : S, XL
Season : Summer
Stock : 27
Minimum Age : 18
Description : Premium quality t-shirt made with comfortable fabric.

----------------------
Score : 0.54
Page  : 11

Name : Black Jeans
Brand : Adidas
Price : I2571
Color : Red
Sizes : S, L
Season : Winter
Stock : 32
Minimum Age : 18
Description : Premium quality shirt made with comfortable fabric.

----------------------
Score : 0.53
Page  : 12

Name : Black Shirt
Brand : Adidas
Price : I1915
Color : Green
Sizes : S, L
Season : Summer
Stock : 18
Minimum Age : 18
Description : Premium quality t-shirt made with comfortable fabric.

----------------------
